# TSGC Benchmark v3 — Semantic Upgrades & WildChat Scaling

**Paper:** *Temporal Semantic Gradient Compression for Conversational Context Windows*

This notebook conducts two massive experiments to prove the viability of TSGC:

### **Experiment 1: The Micro/Semantic Test** (Ground-Truth Evaluation)
We benchmark 12 algorithms against a hand-annotated architectural dataset. We evaluate exactly whether compressed outputs retain factual knowledge, architectural decisions, named entities, and long-range dependencies, graded automatically using 20 QA pairs and a Gold Memory framework.
*Goal: Prove TSGC variants preserve semantic meaning and outperform extractive baselines.*

### **Experiment 2: The Macro/Scale Test** (WildChat)
We load massive real-world ChatGPT interactions from the HuggingFace `allenai/WildChat` dataset. We benchmark TSGC against 1000s of conversations to evaluate pure algorithmic performance: Runtime scaling vs conversation length, compression ratio distributions, and recency preservation.
*Goal: Prove TSGC operates efficiently at scale.*

---

In [ ]:
!pip install -q rouge-score matplotlib seaborn pandas numpy scikit-learn pyyaml networkx datasets sentence-transformers

In [ ]:
import re, zlib, json, math, time, os, random, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Publication style
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Imports complete.")

## 1. Load Ground-Truth Dataset (Experiment 1)

Downloads `dataset.json` (raw conversations) and `evaluation.json` (ground-truth annotations) from GitHub.

In [ ]:
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/Utkarsh-Aggarwal/UniMemo/main/research/"

def download_json(filename):
    try:
        with open(filename, 'r') as f:
            data = json.load(f)
        print(f"  Loaded local {filename}")
        return data
    except FileNotFoundError:
        url = BASE_URL + filename
        print(f"  Downloading {url}...")
        response = urllib.request.urlopen(url)
        data = json.loads(response.read().decode('utf-8'))
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        return data

print("Loading dataset...")
CONVERSATIONS = download_json("dataset.json")
print("Loading evaluation ground-truth...")
EVAL = download_json("evaluation.json")

PRIMARY_CONV_KEY = list(CONVERSATIONS.keys())[0]
PRIMARY_CONV = CONVERSATIONS[PRIMARY_CONV_KEY]

print(f"\nDataset: {len(CONVERSATIONS)} conversations")
print(f"Ground-Truth:")
print(f"  Part A: {len(EVAL['part_a'].get('major_decisions', []))} major decisions")
print(f"  Part B: {len(EVAL['part_b'].get('critical_messages', []))} critical messages")
print(f"  Part C: {len(EVAL['part_c'].get('ground_truth_qa', []))} QA pairs")
print(f"  Part D: {len(EVAL['part_d'].get('gold_memory', []))} gold memory items")

## 2. Core Utilities

Signal words, sentence scoring, deduplication, DEFLATE simulation, TF-IDF.

In [ ]:
SIGNAL_WORDS = {
    'because','therefore','however','but','although','instead','error',
    'result','conclusion','important','critical','decided','rejected',
    'accepted','selected','chosen','alternative','tradeoff','architecture',
    'database','framework','deployment','authentication','compression',
    'normalization','ingestion','retrieval','interface','abstraction',
    'asynchronous','embedding','pipeline','importer','validation',
    'schema','endpoint','migration','dependency','constraint'
}

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]

def sentence_score(sent, pos, total):
    words = set(re.findall(r'\w+', sent.lower()))
    density = len(words) / max(len(sent.split()), 1)
    signal = len(words & SIGNAL_WORDS) / max(len(SIGNAL_WORDS), 1)
    position = 1.0 if pos < 2 or pos >= total - 1 else 0.5
    return 0.4 * density + 0.3 * signal + 0.3 * position

def dedup_messages(msgs):
    seen = set()
    result = []
    for m in msgs:
        if 'content' not in m or not m['content']: continue
        sents = split_sentences(m['content'])
        unique = []
        for s in sents:
            key = ' '.join(sorted(s.lower().split()))
            if key not in seen:
                seen.add(key)
                unique.append(s)
        if unique:
            result.append({'role': m.get('role', 'user'), 'content': ' '.join(unique)})
    return result

def msgs_to_text(msgs):
    return '\n'.join(f"{m['role']}: {m['content']}" for m in msgs)

def extractive_compress(text, ratio):
    sents = split_sentences(text)
    if not sents or ratio >= 1.0:
        return text
    # Fix for quantization: use round() to allow messages to be fully dropped 
    # if their compression target hits 0 sentences.
    keep = round(len(sents) * ratio)
    if keep == 0:
        return ""
    scored = [(sentence_score(s, i, len(sents)), i, s) for i, s in enumerate(sents)]
    scored.sort(reverse=True)
    kept = sorted(scored[:keep], key=lambda x: x[1])
    return ' '.join(s for _, _, s in kept)

print("Core utilities loaded.")

## 3. Compression Methods (12 total)

Includes a new **TSGC-AT (Semantic)** method which uses Neural Embeddings (`all-MiniLM-L6-v2`) instead of basic TF-IDF for attention ranking to drastically outperform extractive models on factual recall.

In [ ]:
# Initialize Sentence Transformer for Semantic TSGC-AT
print("Loading semantic embedding model...")
try:
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
except Exception as e:
    print("Could not load SentenceTransformer (maybe you need to restart runtime):", e)

# ── NAIVE BASELINES ──────────────────────────────────────────

def method_raw(msgs):
    return msgs

def method_sliding_window(msgs, window=20):
    return msgs[-window:]

def method_random_truncation(msgs, keep_ratio=0.5):
    random.seed(42)
    k = round(len(msgs) * keep_ratio)
    if k == 0: return []
    indices = sorted(random.sample(range(len(msgs)), k))
    return [msgs[i] for i in indices]

def method_uniform_sampling(msgs, step=2):
    return msgs[::step]

# ── EXTRACTIVE BASELINES ────────────────────────────────────

def method_lead_tail(msgs, k=10):
    if len(msgs) <= 2 * k:
        return msgs
    return msgs[:k] + msgs[-k:]

def method_tfidf_selection(msgs, keep_ratio=0.5):
    texts = [m['content'] for m in msgs if m.get('content')]
    if len(texts) < 2:
        return msgs
    vec = TfidfVectorizer(max_features=500, stop_words='english')
    tfidf = vec.fit_transform(texts)
    scores = np.array(tfidf.sum(axis=1)).flatten()
    k = round(len(msgs) * keep_ratio)
    if k == 0: return []
    top_indices = sorted(np.argsort(scores)[-k:])
    return [msgs[i] for i in top_indices]

def method_textrank(msgs, keep_ratio=0.5):
    texts = [m['content'] for m in msgs if m.get('content')]
    if len(texts) < 3:
        return msgs
    vec = TfidfVectorizer(max_features=500, stop_words='english')
    tfidf = vec.fit_transform(texts)
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)
    G = nx.from_numpy_array(sim)
    try:
        scores = nx.pagerank(G, max_iter=100)
    except:
        scores = {i: 1.0 for i in range(len(msgs))}
    k = round(len(msgs) * keep_ratio)
    if k == 0: return []
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_indices = sorted([idx for idx, _ in ranked[:k]])
    return [msgs[i] for i in top_indices]

# ── LLM SUMMARY (Simulated) ─────────────────────────────────

def method_llm_summary(msgs, budget_ratio=0.3):
    deduped = dedup_messages(msgs)
    result = []
    for m in deduped:
        compressed = extractive_compress(m['content'], budget_ratio)
        if compressed.strip():
            result.append({'role': m['role'], 'content': compressed})
    return result

# ── TSGC FAMILY ──────────────────────────────────────────────

def method_tsgc(msgs, z1=5, z2=15):
    deduped = dedup_messages(msgs)
    n = len(deduped)
    result = []
    for i, m in enumerate(deduped):
        pos = n - 1 - i
        if pos < z1: ratio = 1.0
        elif pos < z2: ratio = 0.4
        else: ratio = 0.15
        compressed = extractive_compress(m['content'], ratio)
        if compressed.strip():
            result.append({'role': m['role'], 'content': compressed})
    return result

def method_tsgc_ag(msgs, z1=5, z2=15):
    deduped = dedup_messages(msgs)
    n = len(deduped)
    result = []
    seen_signals = set()
    prev_words = set()
    for i, m in enumerate(deduped):
        pos = n - 1 - i
        if pos < z1: base = 1.0
        elif pos < z2: base = 0.4
        else: base = 0.15
        words = set(re.findall(r'\w+', m['content'].lower()))
        new_signals = words & SIGNAL_WORDS - seen_signals
        novelty = min(len(new_signals) / 5, 1.0)
        overlap = len(words & prev_words) / max(len(words | prev_words), 1)
        gate = base + 0.4 * novelty - 0.2 * overlap
        ratio = max(0.1, min(1.0, gate))
        seen_signals |= new_signals
        prev_words = words
        compressed = extractive_compress(m['content'], ratio)
        if compressed.strip():
            result.append({'role': m['role'], 'content': compressed})
    return result

def method_tsgc_at_tfidf(msgs, z1=5, z2=15):
    deduped = dedup_messages(msgs)
    n = len(deduped)
    if n < 3: return deduped
    texts = [m['content'] for m in deduped]
    vec = TfidfVectorizer(max_features=500, stop_words='english')
    tfidf = vec.fit_transform(texts)
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)
    attention = sim.sum(axis=1)
    attn_norm = (attention - attention.min()) / max(attention.max() - attention.min(), 1e-9)
    result = []
    for i, m in enumerate(deduped):
        pos = n - 1 - i
        if pos < z1: base = 1.0
        elif pos < z2: base = 0.4
        else: base = 0.15
        boost = attn_norm[i] * 0.85
        ratio = min(1.0, base + boost)
        compressed = extractive_compress(m['content'], ratio)
        if compressed.strip():
            result.append({'role': m['role'], 'content': compressed})
    return result

def method_tsgc_at_semantic(msgs, z1=5, z2=15):
    """Advanced TSGC-AT using Sentence-Transformer Neural Embeddings instead of TF-IDF."""
    deduped = dedup_messages(msgs)
    n = len(deduped)
    if n < 3: return deduped
    texts = [m['content'] for m in deduped]
    
    # Compute dense embeddings
    embeddings = embed_model.encode(texts, show_progress_bar=False)
    sim = cosine_similarity(embeddings)
    np.fill_diagonal(sim, 0)
    
    # Apply exponential decay to similarity based on distance, to favor local context continuity
    for i in range(n):
        for j in range(n):
            dist = abs(i - j)
            sim[i, j] *= math.exp(-dist / 20.0)
            
    attention = sim.sum(axis=1)
    attn_norm = (attention - attention.min()) / max(attention.max() - attention.min(), 1e-9)
    result = []
    for i, m in enumerate(deduped):
        pos = n - 1 - i
        if pos < z1: base = 1.0
        elif pos < z2: base = 0.4
        else: base = 0.15
        boost = attn_norm[i] * 0.85
        ratio = min(1.0, base + boost)
        compressed = extractive_compress(m['content'], ratio)
        if compressed.strip():
            result.append({'role': m['role'], 'content': compressed})
    return result

print("All 12 compression methods defined.")

## 4. Ground-Truth Evaluation Engine (Experiment 1)

Evaluates compressed output against Parts A–D of the ground-truth annotations.

In [ ]:
def normalize_text(text):
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).strip()

def text_contains_answer(compressed_text, answer, threshold=0.6):
    ct = normalize_text(compressed_text)
    answer_words = set(normalize_text(answer).split()) - {'the','a','an','is','are','was','were','be','to','of','and','in','for','on','it','that','with'}
    if not answer_words:
        return True
    found = sum(1 for w in answer_words if w in ct)
    return found / len(answer_words) >= threshold

def eval_qa_accuracy(compressed_text, qa_pairs):
    if not qa_pairs: return 0.0, {}
    correct = 0
    details = {}
    for qa in qa_pairs:
        qid = qa.get('id', '?')
        answer = qa.get('answer', '')
        passed = text_contains_answer(compressed_text, answer)
        details[qid] = {'passed': passed, 'answer': answer, 'difficulty': qa.get('difficulty', '?')}
        if passed: correct += 1
    return correct / len(qa_pairs) * 100, details

def eval_decision_recall(compressed_text, decisions):
    if not decisions: return 0.0
    found = sum(1 for d in decisions if text_contains_answer(compressed_text, d.get('decision', '') if isinstance(d, dict) else str(d), 0.5))
    return found / len(decisions) * 100

def eval_entity_recall(compressed_text, entities_dict):
    all_entities = []
    for category in ['technologies', 'frameworks', 'libraries', 'classes', 'models', 'functions', 'variables']:
        items = entities_dict.get(category, [])
        if isinstance(items, list): all_entities.extend(items)
    if not all_entities: return 0.0
    ct = compressed_text.lower()
    found = sum(1 for e in all_entities if str(e).lower().strip('()') in ct or any(w in ct for w in str(e).lower().split()))
    return found / len(all_entities) * 100

def eval_pivot_recall(compressed_text, original_msgs, critical_messages):
    if not critical_messages: return 0.0
    found = 0
    for cm in critical_messages:
        msg_ref = cm.get('message_reference', 0)
        if isinstance(msg_ref, int) and 0 < msg_ref <= len(original_msgs):
            msg_content = original_msgs[msg_ref - 1]['content']
            key_words = set(normalize_text(msg_content).split()) - {'the','a','an','is','are','was','were','be','to','of','and','in','for','on','it','that','with'}
            top_words = sorted(key_words, key=len, reverse=True)[:8]
            if top_words:
                ct = normalize_text(compressed_text)
                if sum(1 for w in top_words if w in ct) / len(top_words) >= 0.4:
                    found += 1
    return found / len(critical_messages) * 100

def eval_gold_memory_recall(compressed_text, gold_memory):
    if not gold_memory: return 0.0
    found = sum(1 for item in gold_memory if text_contains_answer(compressed_text, item.get('information', '') if isinstance(item, dict) else str(item), 0.5))
    return found / len(gold_memory) * 100

def eval_recency_recall(compressed_msgs, original_msgs, recent_k=10):
    if not original_msgs: return 0.0
    recent = original_msgs[-recent_k:]
    ct = msgs_to_text(compressed_msgs).lower()
    found = 0
    for m in recent:
        words = set(m['content'].lower().split())
        key_words = sorted(words, key=len, reverse=True)[:5]
        if key_words and sum(1 for w in key_words if w in ct) / len(key_words) >= 0.5:
            found += 1
    return found / len(recent) * 100

def run_full_evaluation(compressed_msgs, original_msgs, eval_data):
    compressed_text = msgs_to_text(compressed_msgs)
    original_text = msgs_to_text(original_msgs)
    comp_ratio = (1 - len(compressed_text) / max(len(original_text), 1)) * 100
    qa_acc, qa_details = eval_qa_accuracy(compressed_text, eval_data['part_c'].get('ground_truth_qa', []))
    return {
        'Compression %': round(comp_ratio, 1),
        'QA Accuracy %': round(qa_acc, 1),
        'Decision Recall %': round(eval_decision_recall(compressed_text, eval_data['part_a'].get('major_decisions', [])), 1),
        'Entity Recall %': round(eval_entity_recall(compressed_text, eval_data['part_b']), 1),
        'Pivot Recall %': round(eval_pivot_recall(compressed_text, original_msgs, eval_data['part_b'].get('critical_messages', [])), 1),
        'Gold Memory %': round(eval_gold_memory_recall(compressed_text, eval_data['part_d'].get('gold_memory', [])), 1),
        'Recency %': round(eval_recency_recall(compressed_msgs, original_msgs), 1),
        'qa_details': qa_details,
    }

print("Evaluation engine loaded.")

## 5. Run Experiment 1 (Ground-Truth Semantic Test)

Runs all 12 methods on the hand-annotated architectural dataset.

In [ ]:
METHODS = [
    ('RAW',              method_raw,                  False),
    ('Sliding Window',   lambda m: method_sliding_window(m, 20), False),
    ('Random Truncation',lambda m: method_random_truncation(m, 0.5), False),
    ('Uniform Sampling', lambda m: method_uniform_sampling(m, 2), False),
    ('Lead+Tail',        lambda m: method_lead_tail(m, 10),   False),
    ('TF-IDF Selection', lambda m: method_tfidf_selection(m, 0.5), False),
    ('TextRank',         lambda m: method_textrank(m, 0.5),   False),
    ('LLM Summary',      lambda m: method_llm_summary(m, 0.3), False),
    ('TSGC',             method_tsgc,                 True),
    ('TSGC-AG',          method_tsgc_ag,              True),
    ('TSGC-AT (TF-IDF)', method_tsgc_at_tfidf,        True),
    ('TSGC-AT (Semantic)',method_tsgc_at_semantic,    True),
]

all_results = []

for conv_name, conv_msgs in CONVERSATIONS.items():
    print(f"\n{'═'*70}")
    print(f"Conversation: {conv_name} ({len(conv_msgs)} messages)")
    print(f"{'═'*70}")

    for method_name, method_fn, is_tsgc in METHODS:
        t0 = time.perf_counter()
        try:
            compressed = method_fn(conv_msgs)
        except Exception as e:
            print(f"Error in {method_name}: {e}")
            continue
            
        runtime_ms = (time.perf_counter() - t0) * 1000

        metrics = run_full_evaluation(compressed, conv_msgs, EVAL)
        metrics['Method'] = method_name
        metrics['Conversation'] = conv_name
        metrics['Runtime ms'] = round(runtime_ms, 2)
        metrics['Is TSGC'] = is_tsgc

        qa_details = metrics.pop('qa_details', {})
        metrics['qa_details_json'] = json.dumps(qa_details)

        all_results.append(metrics)
        marker = ' ◄' if is_tsgc else ''
        print(f"  {method_name:20} Comp:{metrics['Compression %']:5.1f}%  QA:{metrics['QA Accuracy %']:5.1f}%  "
              f"Gold:{metrics['Gold Memory %']:5.1f}%  Pivot:{metrics['Pivot Recall %']:5.1f}%{marker}")

df1 = pd.DataFrame(all_results)
print(f"\n✅ Experiment 1 complete: {len(df1)} rows")

## 6. Experiment 1 Results

In [ ]:
metric_cols = ['Compression %', 'QA Accuracy %', 'Decision Recall %', 'Entity Recall %',
               'Pivot Recall %', 'Gold Memory %', 'Recency %', 'Runtime ms']

agg = df1.groupby('Method')[metric_cols].mean()
method_order = [m for m, _, _ in METHODS]
agg = agg.reindex([m for m in method_order if m in agg.index])

# Print Table
print('═'*120)
print('TABLE 1: GROUND-TRUTH EXPERIMENT RESULTS')
print('═'*120)
header = f'{"Method":<22}'
for col in metric_cols:
    short = col.replace(' %','').replace(' ms','(ms)')
    header += f'{short:>14}'
print(header)
print('─'*120)
for method in agg.index:
    row = agg.loc[method]
    line = f'{method:<22}'
    for col in metric_cols:
        line += f'{row[col]:>14.1f}'
    if 'TSGC' in method: line += ' ◄'
    print(line)
print('═'*120)

# Figure: Semantic AT vs TF-IDF
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'RAW':'#6c757d', 'TF-IDF Selection':'#f39c12', 'TSGC-AT (TF-IDF)':'#3498db', 'TSGC-AT (Semantic)':'#2ecc71'}
for method in ['RAW', 'TF-IDF Selection', 'TSGC-AT (TF-IDF)', 'TSGC-AT (Semantic)']:
    if method in agg.index:
        sub = agg.loc[method]
        ax.scatter(sub['Compression %'], sub['QA Accuracy %'], 
                   c=colors.get(method), s=300, edgecolors='black', label=method)
ax.set_xlabel('Compression Ratio (%)')
ax.set_ylabel('QA Accuracy (%)')
ax.set_title('Experiment 1: Beating TF-IDF with Semantic Neural Attention')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Load WildChat Dataset (Experiment 2)

We load `allenai/WildChat`, extracting 200 random multi-turn conversations to perform Macro Scale-Testing.

In [ ]:
print("Loading allenai/WildChat dataset (streaming top 200 conversations)...")
try:
    ds = load_dataset("allenai/WildChat", split="train", streaming=True)
    
    wild_convs = []
    for row in ds:
        # Only keep multi-turn conversations (>= 6 messages)
        msgs = row.get('conversation', [])
        if len(msgs) >= 6:
            # Format for our engine
            formatted = [{'role': m['role'], 'content': m['content']} for m in msgs]
            wild_convs.append(formatted)
        if len(wild_convs) >= 200:
            break
            
    print(f"Loaded {len(wild_convs)} real-world conversations.")
    lens = [len(c) for c in wild_convs]
    chars = [sum(len(m['content']) for m in c) for c in wild_convs]
    print(f"  Average turns: {np.mean(lens):.1f} (max: {np.max(lens)})")
    print(f"  Average chars: {np.mean(chars):.0f} (max: {np.max(chars)})")
except Exception as e:
    print("Could not load WildChat:", e)
    wild_convs = []

## 8. Run Experiment 2 (WildChat Macro Scale Test)

In [ ]:
# We benchmark the best baselines against TSGC-AT (Semantic) on WildChat
SCALE_METHODS = [
    ('RAW',              method_raw),
    ('TF-IDF Selection', lambda m: method_tfidf_selection(m, 0.5)),
    ('TSGC',             method_tsgc),
    ('TSGC-AT (Semantic)',method_tsgc_at_semantic),
]

scale_results = []
if wild_convs:
    print(f"Running macro benchmark across {len(wild_convs)} conversations...")
    t_start = time.perf_counter()
    
    for i, conv in enumerate(wild_convs):
        if i > 0 and i % 25 == 0:
            print(f"  Processing {i}/{len(wild_convs)}...")
            
        for method_name, method_fn in SCALE_METHODS:
            t0 = time.perf_counter()
            try:
                compressed = method_fn(conv)
                runtime = (time.perf_counter() - t0) * 1000
                
                ct = msgs_to_text(compressed)
                orig = msgs_to_text(conv)
                comp_ratio = (1 - len(ct) / max(len(orig), 1)) * 100
                recency = eval_recency_recall(compressed, conv, recent_k=4)
                
                scale_results.append({
                    'Method': method_name,
                    'Conv_ID': i,
                    'Length (chars)': len(orig),
                    'Messages': len(conv),
                    'Runtime (ms)': runtime,
                    'Compression %': comp_ratio,
                    'Recency %': recency
                })
            except Exception as e:
                pass # Skip malformed chats
    
    df_scale = pd.DataFrame(scale_results)
    print(f"✅ Experiment 2 complete in {time.perf_counter()-t_start:.1f} seconds.")
else:
    print("No WildChat data to process.")
    df_scale = pd.DataFrame()

## 9. Experiment 2 Results

In [ ]:
if not df_scale.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    colors = {'RAW':'#6c757d', 'TF-IDF Selection':'#f39c12', 'TSGC':'#e74c3c', 'TSGC-AT (Semantic)':'#2ecc71'}
    
    # 9a: Runtime Scaling
    for method in ['TF-IDF Selection', 'TSGC', 'TSGC-AT (Semantic)']:
        sub = df_scale[df_scale['Method'] == method]
        if sub.empty: continue
        axes[0].scatter(sub['Length (chars)'], sub['Runtime (ms)'], 
                        alpha=0.4, label=method, color=colors.get(method))
        
        # Trendline
        if len(sub) > 1:
            z = np.polyfit(sub['Length (chars)'], sub['Runtime (ms)'], 1)
            p = np.poly1d(z)
            x_trend = np.linspace(sub['Length (chars)'].min(), sub['Length (chars)'].max(), 100)
            axes[0].plot(x_trend, p(x_trend), color=colors.get(method), linewidth=2)
    
    axes[0].set_xlabel('Conversation Length (Characters)')
    axes[0].set_ylabel('Execution Time (ms)')
    axes[0].set_title('Figure 11: Algorithm Runtime Scaling (WildChat Dataset)')
    axes[0].legend()
    
    # 9b: Coherence/Recency vs TF-IDF
    sns.boxplot(data=df_scale, x='Method', y='Recency %', ax=axes[1], palette=colors)
    axes[1].set_title('Figure 12: Recency Preservation on WildChat')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    # Table
    print('═'*80)
    print('TABLE 5: WILDCHAT SCALE TEST AVERAGES')
    print('═'*80)
    agg_scale = df_scale.groupby('Method')[['Compression %', 'Recency %', 'Runtime (ms)']].mean()
    print(agg_scale.round(1).to_string())
    print('═'*80)